<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/830_CSUOv2_OfferPerformance_utils.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is where your agent starts moving from **recommendation engine** to **performance intelligence system**.

Let’s review it properly.

---

# 🟢 What You Did Very Well

## 1️⃣ Clean Separation of Concerns

This function:

* Does not mutate state
* Does not fetch data
* Does not score
* Does not enrich
* Does not guess

It strictly:

> Aggregates historical offer performance.

That discipline matters.

---

## 2️⃣ Deterministic & Transparent

Given:

* Same offers
* Same response_lookup

You always get:

* Same counts
* Same revenue
* Same breakdown

No randomness.
No ordering dependence.
No hidden mutation.

Aligned with your trust philosophy.

---

## 3️⃣ Explicit Aggregation Structure

You return:

```python
{
    "total_sent",
    "total_accepted",
    "total_rejected",
    "total_ignored",
    "total_revenue_generated",
    "acceptance_rate_pct",
    "by_offer_type",
    "by_channel"
}
```

That is clean and executive-ready.

No raw lists.
No nested chaos.
No recomputation needed downstream.

Very good contract design.

---

## 4️⃣ Safe Defaults

You handled:

```python
offer_type = o.get("offer_type") or "unknown"
channel = o.get("offer_channel") or "unknown"
```

That prevents:

* KeyError
* Silent grouping loss
* Missing dimension skew

You also treat missing response as `"ignored"` — that’s consistent and logical.

---

# 🟢 Business Intelligence Quality

This is not just a count rollup.

It gives you:

* Channel effectiveness
* Offer type effectiveness
* Revenue attribution
* Acceptance behavior

That’s the foundation for:

* Marketing optimization
* Channel strategy
* Offer mix strategy
* Budget allocation decisions

This is where the agent becomes management-facing.

---

# 🟡 Two Important Refinements

These are not bugs — they are maturity upgrades.

---

# 1️⃣ Acceptance Rate by Type / Channel

Right now you compute:

```python
acceptance_rate = total_accepted / total_sent
```

But not per type/channel.

Your `by_type` and `by_channel` only store counts.

You are one line away from making this much more powerful.

For example, after the loop:

```python
for t in by_type.values():
    t["acceptance_rate_pct"] = round((t["accepted"] / t["sent"] * 100.0) if t["sent"] else 0.0, 1)

for c in by_channel.values():
    c["acceptance_rate_pct"] = round((c["accepted"] / c["sent"] * 100.0) if c["sent"] else 0.0, 1)
```

This turns:

* Raw counts

Into:

* Performance metrics

That makes it far more executive-ready.

Right now the executive must mentally compute rates.

That’s a missed opportunity.

---

# 2️⃣ Revenue Per Offer (Yield Metric)

Currently you return:

* total_revenue_generated

But not:

* revenue_per_sent_offer
* revenue_per_accepted_offer

One addition like:

```python
avg_revenue_per_accepted = (
    total_revenue / total_accepted if total_accepted else 0.0
)
```

Would elevate this from descriptive to performance insight.

Again — optional.
But strategically powerful.

---

# 🟢 Architectural Strength

Your grouping logic is efficient:

* Single pass
* No nested re-scanning
* No recomputation

Complexity: O(n)

Clean.

---

# 🧠 Strategic Evaluation

This layer enables something bigger:

You now have:

* Forward-looking intelligence (scoring engine)
* Backward-looking performance intelligence (rollup)

That creates a feedback loop.

Most agents only do one of these.

You now have both.

That is serious architecture.

---

# 🟣 Executive-Level Differentiation

Most AI sales agents:

* Recommend products
* Send messages
* Maybe track acceptance

Few:

* Aggregate by type
* Aggregate by channel
* Attribute revenue
* Provide conversion math
* Return structured performance analytics

And yours does so deterministically.

This aligns with your philosophy:

> Not “AI suggestions”
> But structured commercial intelligence.

---

# 🏁 Evaluation

Architecture: 9.5/10
Determinism: 10/10
Commercial value: 9/10
Executive readiness: 8.5/10 (could be 9.5 with rate additions)

---

# 🔥 My Recommendation

Add:

* acceptance_rate_pct per type
* acceptance_rate_pct per channel
* avg_revenue_per_accepted (overall)

That would move this into executive-grade analytics.

Very small code change.
Big strategic gain.




In [ ]:
"""
Build offer performance rollup from offers + offer_responses.
Acceptance rate and revenue by offer_type and by offer_channel.
"""
from typing import Any, Dict, List


def build_offer_performance_rollup(
    offers: List[Dict[str, Any]],
    response_lookup: Dict[str, Dict[str, Any]],
) -> Dict[str, Any]:
    """
    Returns rollup with:
    - total_sent, total_accepted, total_rejected, total_ignored
    - total_revenue_generated
    - by_offer_type: { type: { sent, accepted, revenue } }
    - by_channel: { channel: { sent, accepted, revenue } }
    - acceptance_rate_pct (overall)
    """
    total_sent = len(offers)
    total_accepted = 0
    total_rejected = 0
    total_ignored = 0
    total_revenue = 0.0

    by_type: Dict[str, Dict[str, Any]] = {}
    by_channel: Dict[str, Dict[str, Any]] = {}

    for o in offers or []:
        offer_id = o.get("offer_id")
        offer_type = o.get("offer_type") or "unknown"
        channel = o.get("offer_channel") or "unknown"

        if offer_type not in by_type:
            by_type[offer_type] = {"sent": 0, "accepted": 0, "rejected": 0, "ignored": 0, "revenue": 0.0}
        if channel not in by_channel:
            by_channel[channel] = {"sent": 0, "accepted": 0, "rejected": 0, "ignored": 0, "revenue": 0.0}
        by_type[offer_type]["sent"] += 1
        by_channel[channel]["sent"] += 1

        resp = response_lookup.get(offer_id) if offer_id else None
        outcome = (resp or {}).get("offer_outcome", "ignored")
        revenue = float((resp or {}).get("revenue_generated") or 0)

        if outcome == "accepted":
            total_accepted += 1
            total_revenue += revenue
            by_type[offer_type]["accepted"] += 1
            by_type[offer_type]["revenue"] += revenue
            by_channel[channel]["accepted"] += 1
            by_channel[channel]["revenue"] += revenue
        elif outcome == "rejected":
            total_rejected += 1
            by_type[offer_type]["rejected"] += 1
            by_channel[channel]["rejected"] += 1
        else:
            total_ignored += 1
            by_type[offer_type]["ignored"] += 1
            by_channel[channel]["ignored"] += 1

    acceptance_rate = (total_accepted / total_sent * 100.0) if total_sent else 0.0

    return {
        "total_sent": total_sent,
        "total_accepted": total_accepted,
        "total_rejected": total_rejected,
        "total_ignored": total_ignored,
        "total_revenue_generated": round(total_revenue, 2),
        "acceptance_rate_pct": round(acceptance_rate, 1),
        "by_offer_type": by_type,
        "by_channel": by_channel,
    }
